In [23]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd
import re
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [24]:
city_urls = {
    "Hyderabad": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Hyderabad",
    "Bangalore": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Bangalore",
    "Mumbai": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Mumbai",
    "Pune": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Pune",
    "Chennai": "https://www.magicbricks.com/property-for-rent/residential-real-estate?cityName=Chennai"
}

# Define headers to prevent a browser from blocking
headers = {'User-Agent': 'Mozilla/5.0'}

max_properties_per_city = 300

# list to store all scraped data
all_data = []

# Keywords to detect actual property types from title text
property_keywords = ['Apartment', 'Flat', 'Studio', 'Villa', 'House', 'Floor', 'Penthouse', 'Row House', 'Farm']

# Loop through each city and its corresponding URL
for city, base_url in city_urls.items():
    print(f"\nScraping {city}...")
    city_data = []   # Store city-specific listings
    page = 1      # Start from page 1

    # Loop through pages until desired property count is reached
    while len(city_data) < max_properties_per_city:
        url = f"{base_url}&page={page}"
        res = requests.get(url, headers=headers)
        soup = BeautifulSoup(res.text, 'html.parser')

        # Find all property listing cards on the page
        listings = soup.find_all("div", class_="mb-srp__card")
        if not listings:
            print(" No listings found, stopping.")
            break

        # Loop through each property card
        for listing in listings:
            try:
                # Extract bhk from title
                title_tag = listing.find("h2", class_="mb-srp__card--title")
                title = title_tag.get_text(strip=True)

                bhk_match = re.search(r"(\d+)\s*BHK", title.upper())
                bhk = bhk_match.group(1) if bhk_match else None

                # Extract location
                locality = title.split("in", 1)[-1].strip() if "in" in title else None

                # Detect property type from title
                title_clean = title.replace(",", " ").replace("-", " ")
                title_words = title_clean.split()
                prop_type_from_title = next((word for word in title_words if word.capitalize() in property_keywords), None)
            except:
                bhk = locality = prop_type_from_title = None

            # Extract rent price
            try:
                rent = listing.find("div", class_="mb-srp__card__price--amount").get_text(strip=True)
                rent = rent.replace("₹", "").replace(",", "").strip()
            except:
                rent = None

            summary = listing.find("div", class_="mb-srp__card__summary__list")
            summary_items = summary.find_all("div", class_="mb-srp__card__summary__list--item") if summary else []

            # Initializing Variable
            area = furnishing = facing = property_type = overlooking = Bathroom = Balcony=TenantPreferred=Availability= None
            
            # Loop through summary items and extract specific features
            for item in summary_items:
                try:
                    label = item.find("div", class_="mb-srp__card__summary--label").get_text(strip=True).lower()
                    value = item.find("div", class_="mb-srp__card__summary--value").get_text(strip=True)

                    if "area" in label:
                        area = value
                    elif "furnish" in label:
                        furnishing = value
                    elif "facing" in label:
                        facing = value
                    elif "property type" in label or "type" in label:
                        property_type = value
                    elif "overlooking" in label :
                        overlooking = value
                    elif "bathroom" in label:
                        Bathroom = value  
                    elif "balcony" in label:
                        Balcony = value 
                    elif "tenant preferred" in label:
                        TenantPreferred=value
                    elif "availability" in label:
                        Availability=value
                        

                except:
                    continue

            # Final fallback if property_type is missing or invalid
            if not property_type or property_type.lower() in ["for", "in", "rent", "sale"]:
                property_type = prop_type_from_title

            if not bhk or not locality:
                continue

            city_data.append({
                "City": city,
                "BHK": bhk.strip(),
                "Location": locality.strip(),
                "Price (₹)": rent,
                "Area (sqft)": area.strip() if area else None,
                "Property Type": property_type.strip() if property_type else None,
                "Furnishing": furnishing.strip() if furnishing else None,
                "Property Facing": facing.strip() if facing else None,
                "overlooking":overlooking.strip() if overlooking else None,
                "Bathroom":Bathroom.strip() if Bathroom else None,
                "Balcony":Balcony.strip() if Balcony else None,
                "Tenant Preferred":TenantPreferred.strip() if TenantPreferred else None,
                "Availability":Availability.strip() if Availability else None,

                
            })

            if len(city_data) >= max_properties_per_city:
                break

        page += 1    # Move to next Page
        time.sleep(1)  # delay to avoid overloading the server


    # Append city data to the master list
    all_data.extend(city_data)
print("\nScraping completed...")


Scraping Hyderabad...

Scraping Bangalore...

Scraping Mumbai...

Scraping Pune...

Scraping Chennai...

Scraping completed...


In [25]:
df = pd.DataFrame(all_data)
df.to_csv("MagicBricksProject_Scraped_Data.csv", index=False)
print("\nData saved to MagicBricksProject_Scraped_Data.csv")


Data saved to MagicBricksProject_Scraped_Data.csv


In [26]:
df=pd.read_csv("MagicBricksProject_Scraped_Data.csv")

In [27]:
df

,City,BHK,Location,Price (₹),Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,1,"Kondapur, Hyderabad",25000,400 sqft,Flat,Furnished,East,Main Road,1,1.0,Bachelors/Family,Immediately
1,Hyderabad,3,"Prestige Tranquil, Kokapet, Outer Ring Road, H...",70000,1361 sqft,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,1.0,NaN,NaN
2,Hyderabad,4,"Kokapet, Outer Ring Road Hyderabad",3.1 Lac,5100 sqft,Villa,Semi-Furnished,East,"Pool, Main Road",4,1.0,Bachelors,Immediately
3,Hyderabad,3,"SH Casa Rouge, Kondapur, Hyderabad",70000,1946 sqft,Flat,Furnished,West,Garden/Park,3,1.0,Bachelors/Family,Immediately
4,Hyderabad,3,"Prestige Tranquil, Kokapet, Outer Ring Road, H...",70000,1168 sqft,Flat,Semi-Furnished,West,"Pool, Garden/Park, Main Road",3,1.0,Bachelors,Immediately
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,Chennai,3,"House of Hiranandani, Egattur, Chennai",1 Lac,2170 sqft,Flat,Semi-Furnished,South - East,Garden/Park,3,2.0,Bachelors/Family,Immediately
1496,Chennai,5,Thoraipakkam Chennai,14500,1250 sqft,Villa,Furnished,NaN,NaN,4,2.0,Bachelors/Family,Immediately
1497,Chennai,2,"Villivakkam, Chennai",11000,845 sqft,Flat,Furnished,NaN,NaN,2,NaN,Bachelors/Family,Immediately
1498,Chennai,1,"West Saidapet, Chennai",12000,480 sqft,Flat,Unfurnished,East,Main Road,1,1.0,Bachelors/Family,Immediately


In [28]:

# prints number of columns
print("Number of columns:",df.shape[1])

Number of columns: 13


In [29]:

# prints number of rows
print("Number of rows:",df.shape[0])

Number of rows: 1500


In [30]:
# Print the data types of each column
print("Data types of each column:\n",df.dtypes)

Data types of each column:
 City                 object
BHK                   int64
Location             object
Price (₹)            object
Area (sqft)          object
Property Type        object
Furnishing           object
Property Facing      object
overlooking          object
Bathroom             object
Balcony             float64
Tenant Preferred     object
Availability         object
dtype: object


In [31]:
# Print the number of missing values in each column
print("Missing values in each column:\n",df.isnull().sum())

Missing values in each column:
 City                  0
BHK                   0
Location              0
Price (₹)             0
Area (sqft)           1
Property Type         0
Furnishing            6
Property Facing     203
overlooking         288
Bathroom              1
Balcony             278
Tenant Preferred      6
Availability          6
dtype: int64


In [32]:
# It shows the duplicated rows count
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 0


In [33]:
df.drop_duplicates(subset=['City', 'BHK', 'Location', 'Price (₹)', 'Area (sqft)', 'Property Type', 'Furnishing', 'Property Facing'
,'overlooking','Bathroom','Balcony','Tenant Preferred','Availability'], inplace=True)

In [34]:
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 0


In [35]:
print("Number of rows:",df.shape[0])

Number of rows: 1500


In [36]:
# List of locations 
localities = [
    "Kondapur", "Narsingi", "Gachibowli", "Tellapur", "Kompally", "Kollur", "Shamshabad","Nalagandla",
    "Bachupally", "Miyapur", "Hitech City", "Puppalaguda", "Kokapet", "Manikonda", "Madhapur",
    "Whitefield", "Varthur", "Hebbal", "Yelahanka", "Devanahalli",
    "Sarjapur", "Electronic City", "Kanakapura Road", "Bannerghatta Main Road","Rajajinagar",
    "Mulund", "Andheri", "Chembur", "Worli", "Powai", "Borivali", "Bandra", "Wadala",
    "Goregaon", "Kandivali East", "Thakur Village", "Juhu", "Kandivali", "Hadapsar", "Mundhwa",
    "Kharadi", "Baner", "Hadapsar", "Hinjawadi", "Wagholi", "Wakad", "Balewadi",
    "NIBM Road", "Viman Nagar", "Undri", "Sholinganallur", "Medavakkam", "Pallikaranai", "Porur", "Perungudi", 
    "Pallavaram", "Guduvancheri", "Kelambakkam", "Thiruvanmiyur", "Padur"
    # ---------------- PUNE ----------------
    "Shivaji Nagar", "Deccan Gymkhana", "Camp", "Swargate", "Kothrud", "Aundh",
    "Baner", "Bavdhan", "Pashan", "Warje", "Viman Nagar", "Koregaon Park",
    "Kalyani Nagar", "Kharadi", "Hadapsar", "Wagholi", "Mundhwa", "Bibwewadi",
    "Katraj", "Dhayari", "Hinjewadi", "Wakad", "Pimple Saudagar", "Pimpri",

    # ---------------- HYDERABAD ----------------
    "Charminar", "Malakpet", "Nampally", "Asif Nagar", "Gachibowli",
    "HITEC City", "Madhapur", "Kondapur", "Financial District", "Nanakramguda",
    "Kukatpally", "Miyapur", "Chanda Nagar", "Hafeezpet", "Bachupally",
    "Nizampet", "Manikonda", "Narsingi", "Uppal", "Begumpet",
    "Banjara Hills", "Jubilee Hills", "Punjagutta", "Secunderabad",

    # ---------------- MUMBAI ----------------
    "Andheri", "Andheri East", "Andheri West", "Bandra", "Bandra East", "Bandra West",
    "Borivali", "Borivali East", "Borivali West", "Dadar", "Dadar East", "Dadar West",
    "Goregaon", "Goregaon East", "Goregaon West", "Juhu", "Kandivali",
    "Kandivali East", "Kandivali West", "Malad", "Malad East", "Malad West",
    "Lower Parel", "Worli", "Powai", "Chembur", "Ghatkopar", "Kurla",
    "Thane", "Navi Mumbai", "Vashi", "Nerul", "Kharghar",

    # ---------------- CHENNAI ----------------
    "T Nagar", "Adyar", "Velachery", "Anna Nagar", "Tambaram", "Chromepet",
    "Guindy", "Porur", "Vadapalani", "Nungambakkam", "Mylapore", "Kodambakkam",
    "Royapettah", "Egmore", "Triplicane", "Thiruvanmiyur", "Perungudi",
    "Sholinganallur", "OMR", "ECR", "Pallavaram", "Medavakkam",

    # ---------------- BANGALORE ----------------
    "Whitefield", "Marathahalli", "KR Puram", "Indiranagar", "Koramangala",
    "HSR Layout", "BTM Layout", "Electronic City", "Bellandur", "Sarjapur Road",
    "MG Road", "Jayanagar", "JP Nagar", "Banashankari", "Yelahanka",
    "Hebbal", "Malleshwaram", "Rajajinagar", "Basavanagudi", "Kengeri",
    "Nagarbhavi", "Vijayanagar"
]

# Lowercased for comparison
localities_lower = [loc.lower() for loc in localities]

# Function to clean location
def clean_location(text):
    if pd.isna(text):
        return ''
    text = re.sub(r'\s+', ' ', text).lower()
    for loc, loc_lower in zip(localities, localities_lower):
        if loc_lower in text:
            return loc
    return np.nan

# Update Location column in-place
df['Location'] = df['Location'].apply(clean_location)  

In [37]:
df["Location"] = df["Location"].replace("Hinjawadi", "Hinjewadi")

In [38]:
# renaming the column
df.rename(columns={"Price (₹)": "Price"}, inplace=True)

In [39]:
# Function to clean and convert price to integer
def clean_price(price):
    price = str(price).replace(',', '').strip().lower()

    if 'lac' in price:
        num = float(price.split()[0])
        return int(round(num * 100000))
    elif 'cr' in price:
        num = float(price.split()[0])
        return int(round(num * 10000000))
    else:
        return int(''.join(filter(str.isdigit, price)))
    
# Apply cleaning function to 'Price' column
df["Price"] = df["Price"].apply(clean_price)

In [40]:
# Extract numeric part from 'Area (sqft)' and convert to integer
df["Area (sqft)"] = pd.to_numeric(df["Area (sqft)"].astype(str).str.extract(r'(\d+)', expand=False), errors='coerce').fillna(0).astype(int)

In [41]:
df.head(10)

,City,BHK,Location,Price,Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,1,Kondapur,25000,400,Flat,Furnished,East,Main Road,1,1.0,Bachelors/Family,Immediately
1,Hyderabad,3,Kokapet,70000,1361,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,1.0,NaN,NaN
2,Hyderabad,4,Kokapet,310000,5100,Villa,Semi-Furnished,East,"Pool, Main Road",4,1.0,Bachelors,Immediately
3,Hyderabad,3,Kondapur,70000,1946,Flat,Furnished,West,Garden/Park,3,1.0,Bachelors/Family,Immediately
4,Hyderabad,3,Kokapet,70000,1168,Flat,Semi-Furnished,West,"Pool, Garden/Park, Main Road",3,1.0,Bachelors,Immediately
5,Hyderabad,3,Kondapur,45000,1495,Flat,Semi-Furnished,East,"Garden/Park, Pool, Main Road",3,1.0,Bachelors/Family,Immediately
6,Hyderabad,5,Financial District,300000,5000,Flat,Furnished,East,"Garden/Park, Pool",5,3.0,Bachelors/Family,Immediately
7,Hyderabad,4,NaN,300000,2800,Flat,Furnished,West,Garden/Park,4,2.0,Bachelors/Family,Immediately
8,Hyderabad,3,Kokapet,70000,1874,Flat,Semi-Furnished,West,Main Road,3,2.0,Bachelors,Immediately
9,Hyderabad,3,Narsingi,55000,1388,Flat,Semi-Furnished,North - East,Garden/Park,3,1.0,Bachelors/Family,Immediately


In [42]:
# Replace empty strings in 'Location' with NaN
df['Location'] = df['Location'].replace('', np.nan)

In [43]:
# Fill missing 'Location' values with the most frequent location (mode) within each city
df['Location'] = df.groupby('City')['Location'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'Unknown')
)

In [44]:
# Count remaining missing values in 'Location' column
df['Location'].isna().sum()

np.int64(0)

In [45]:
# Fill missing 'Property Facing' values with the most frequent location (mode) within each city
df['Property Facing'] = df.groupby('Location')['Property Facing'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else 'East')
)

In [46]:

# Count remaining missing values in 'Location' column
df['Property Facing'].isna().sum()

np.int64(0)

In [47]:
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 0


In [48]:
df.drop_duplicates(subset=['City', 'BHK', 'Location', 'Price', 'Area (sqft)', 'Property Type', 'Furnishing', 'Property Facing'
,'overlooking','Bathroom','Balcony','Tenant Preferred','Availability'], inplace=True)

In [49]:
num_duplicates = df.duplicated().sum()
print(f"Total duplicate rows (all columns): {num_duplicates}")

Total duplicate rows (all columns): 0


In [50]:
print("Missing values in each column:\n",df.isnull().sum())

Missing values in each column:
 City                  0
BHK                   0
Location              0
Price                 0
Area (sqft)           0
Property Type         0
Furnishing            6
Property Facing       0
overlooking         288
Bathroom              1
Balcony             278
Tenant Preferred      6
Availability          6
dtype: int64


In [51]:
# Fill missing 'Furnishing' values with the most frequent location (mode) within each city
df['Furnishing'] = df['Furnishing'].fillna(df['Furnishing'].mode()[0])

In [52]:
df["overlooking"]=df["overlooking"].fillna(df["overlooking"].mode()[0])

In [53]:
df["Bathroom"]=df["Bathroom"].fillna(df["Bathroom"].mode()[0])

In [54]:
df["Tenant Preferred"]=df["Tenant Preferred"].fillna(df["Tenant Preferred"].mode()[0])

In [55]:
df['Availability']=df['Availability'].fillna(df['Availability'].mode()[0])

In [56]:
df['Balcony']=df['Balcony'].fillna(df['Balcony'].mode()[0])

In [57]:

print("Missing values in each column:\n",df.isnull().sum())

Missing values in each column:
 City                0
BHK                 0
Location            0
Price               0
Area (sqft)         0
Property Type       0
Furnishing          0
Property Facing     0
overlooking         0
Bathroom            0
Balcony             0
Tenant Preferred    0
Availability        0
dtype: int64


In [58]:
print("Data types of each column:\n",df.dtypes)

Data types of each column:
 City                 object
BHK                   int64
Location             object
Price                 int64
Area (sqft)           int64
Property Type        object
Furnishing           object
Property Facing      object
overlooking          object
Bathroom             object
Balcony             float64
Tenant Preferred     object
Availability         object
dtype: object


In [59]:

df['BHK'] = df['BHK'].astype(int)


In [60]:
df

,City,BHK,Location,Price,Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,1,Kondapur,25000,400,Flat,Furnished,East,Main Road,1,1.0,Bachelors/Family,Immediately
1,Hyderabad,3,Kokapet,70000,1361,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,1.0,Bachelors/Family,Immediately
2,Hyderabad,4,Kokapet,310000,5100,Villa,Semi-Furnished,East,"Pool, Main Road",4,1.0,Bachelors,Immediately
3,Hyderabad,3,Kondapur,70000,1946,Flat,Furnished,West,Garden/Park,3,1.0,Bachelors/Family,Immediately
4,Hyderabad,3,Kokapet,70000,1168,Flat,Semi-Furnished,West,"Pool, Garden/Park, Main Road",3,1.0,Bachelors,Immediately
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1495,Chennai,3,Adyar,100000,2170,Flat,Semi-Furnished,South - East,Garden/Park,3,2.0,Bachelors/Family,Immediately
1496,Chennai,5,Adyar,14500,1250,Villa,Furnished,East,Garden/Park,4,2.0,Bachelors/Family,Immediately
1497,Chennai,2,Adyar,11000,845,Flat,Furnished,East,Garden/Park,2,2.0,Bachelors/Family,Immediately
1498,Chennai,1,Adyar,12000,480,Flat,Unfurnished,East,Main Road,1,1.0,Bachelors/Family,Immediately


In [61]:
print("Data types of each column:\n",df.dtypes)

Data types of each column:
 City                 object
BHK                   int64
Location             object
Price                 int64
Area (sqft)           int64
Property Type        object
Furnishing           object
Property Facing      object
overlooking          object
Bathroom             object
Balcony             float64
Tenant Preferred     object
Availability         object
dtype: object


In [62]:
print(df['Price'].describe())
print(df['Area (sqft)'].describe())
print(df['BHK'].describe())
print(df['Bathroom'].describe())
print(df['Balcony'].describe())

count    1.500000e+03
mean     1.213537e+05
std      6.326944e+05
min      5.000000e+03
25%      3.600000e+04
50%      6.175000e+04
75%      1.100000e+05
max      2.300000e+07
Name: Price, dtype: float64
count      1500.000000
mean       1744.238667
std        4037.390856
min           0.000000
25%         909.500000
50%        1335.000000
75%        1993.750000
max      150000.000000
Name: Area (sqft), dtype: float64
count    1500.000000
mean        2.826000
std         0.954462
min         1.000000
25%         2.000000
50%         3.000000
75%         3.000000
max        10.000000
Name: BHK, dtype: float64
count     1500
unique      10
top          3
freq       586
Name: Bathroom, dtype: object
count    1500.000000
mean        1.914000
std         0.801061
min         1.000000
25%         1.000000
50%         2.000000
75%         2.000000
max         7.000000
Name: Balcony, dtype: float64


In [63]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile) of 'Price'
Q1 = df['Price'].quantile(0.25)
Q3 = df['Price'].quantile(0.75)

# Compute the Interquartile Range (IQR)
IQR = Q3 - Q1
# Define upper bound for outliers (1.5 * IQR rule)
price_upper = Q3 + 1.5 * IQR

# Count number of outliers above the upper bound
price_outliers_count = df[df['Price'] > price_upper].shape[0]
print("Number of Price outliers:", price_outliers_count)

Number of Price outliers: 154


In [64]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile) of 'Area'
Q1 = df['Area (sqft)'].quantile(0.25)
Q3 = df['Area (sqft)'].quantile(0.75)

# Compute the Interquartile Range (IQR)
IQR = Q3 - Q1
# Define upper bound for outliers (1.5 * IQR rule)
area_upper = Q3 + 1.5 * IQR

# Count number of outliers above the upper bound
area_outliers_count = df[df['Area (sqft)'] > area_upper].shape[0]
print("Number of Area (sqft) outliers:", area_outliers_count)

Number of Area (sqft) outliers: 83


In [65]:
# Calculate Q1 (25th percentile) and Q3 (75th percentile) of 'bhk'
Q1 = df['BHK'].quantile(0.25)
Q3 = df['BHK'].quantile(0.75)

# Compute the Interquartile Range (IQR)
IQR = Q3 - Q1
# Define upper bound for outliers (1.5 * IQR rule)
bhk_upper = Q3 + 1.5 * IQR

# Count number of outliers above the upper bound
bhk_outliers_count = df[df['BHK'] > bhk_upper].shape[0]
print("Number of BHK outliers:", bhk_outliers_count)

Number of BHK outliers: 51


In [66]:
df = df[df['BHK'] < 5]  # remove BHK > 5

# Remove Price Outliers using IQR 
Q1_price = df['Price'].quantile(0.25)
Q3_price = df['Price'].quantile(0.75)
IQR_price = Q3_price - Q1_price
upper_price = Q3_price + 1.5 * IQR_price

# Keep only rows with Price within the upper limit
df = df[df['Price'] <= upper_price]

# Remove Area Outliers using IQR
Q1_area = df['Area (sqft)'].quantile(0.25)
Q3_area = df['Area (sqft)'].quantile(0.75)
IQR_area = Q3_area - Q1_area
upper_area = Q3_area + 1.5 * IQR_area

# Keep only rows with Price within the upper limit
df = df[df['Area (sqft)'] <= upper_area]

In [67]:
# Clean 'Bathroom' and 'Balcony' by removing non-numeric characters and converting to numeric
df['Bathroom'] = pd.to_numeric(df['Bathroom'].astype(str).str.extract('(\d+)', expand=False), errors='coerce')
df['Balcony'] = pd.to_numeric(df['Balcony'].astype(str).str.extract('(\d+)', expand=False), errors='coerce')

df["Bathroom"] = df["Bathroom"].astype(int)

# Remove Area Outliers using IQR
Q1_area = df['Bathroom'].quantile(0.25)
Q3_area = df['Bathroom'].quantile(0.75)
IQR_area = Q3_area - Q1_area
upper_area = Q3_area + 1.5 * IQR_area

# Keep only rows with Price within the upper limit
df = df[df['Bathroom'] <= upper_area]
print(df['Bathroom'].unique())

[1 3 2 4]


In [68]:
df["Balcony"] = df["Balcony"].astype(int)

# Remove Area Outliers using IQR
Q1_area = df['Balcony'].quantile(0.25)
Q3_area = df['Balcony'].quantile(0.75)
IQR_area = Q3_area - Q1_area
upper_area = Q3_area + 1.5 * IQR_area

# Keep only rows with Price within the upper limit
df = df[df['Balcony'] <= upper_area]
print(df['Balcony'].unique())

[1 2 3]


In [70]:
df.to_csv("MagicBricksProject_Cleaned_Data.csv", index=False)
print("\nData saved to MagicBricksProject_Cleaned_Data.csv")


Data saved to MagicBricksProject_Cleaned_Data.csv


In [71]:
df

,City,BHK,Location,Price,Area (sqft),Property Type,Furnishing,Property Facing,overlooking,Bathroom,Balcony,Tenant Preferred,Availability
0,Hyderabad,1,Kondapur,25000,400,Flat,Furnished,East,Main Road,1,1,Bachelors/Family,Immediately
1,Hyderabad,3,Kokapet,70000,1361,Flat,Semi-Furnished,East,"Pool, Garden/Park, Main Road",3,1,Bachelors/Family,Immediately
3,Hyderabad,3,Kondapur,70000,1946,Flat,Furnished,West,Garden/Park,3,1,Bachelors/Family,Immediately
4,Hyderabad,3,Kokapet,70000,1168,Flat,Semi-Furnished,West,"Pool, Garden/Park, Main Road",3,1,Bachelors,Immediately
5,Hyderabad,3,Kondapur,45000,1495,Flat,Semi-Furnished,East,"Garden/Park, Pool, Main Road",3,1,Bachelors/Family,Immediately
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1494,Chennai,3,Adyar,20000,2025,Villa,Semi-Furnished,North,Garden/Park,3,1,Family,Immediately
1495,Chennai,3,Adyar,100000,2170,Flat,Semi-Furnished,South - East,Garden/Park,3,2,Bachelors/Family,Immediately
1497,Chennai,2,Adyar,11000,845,Flat,Furnished,East,Garden/Park,2,2,Bachelors/Family,Immediately
1498,Chennai,1,Adyar,12000,480,Flat,Unfurnished,East,Main Road,1,1,Bachelors/Family,Immediately
